In [9]:
#Compute slope and then export slope, ogdp, elevation, and protected for each pixel

In [10]:
#Compute Slope
arcpy.env.overwriteOutput = True

with arcpy.EnvManager(scratchWorkspace=r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\gis_data\explore_data\explore_data.gdb"):
    slope_raster = arcpy.sa.Slope(
        in_raster="wetopo2022_albers_10k",
        output_measurement="DEGREE",
        z_factor=1,
        method="PLANAR",
        z_unit="METER",
        analysis_target_device="GPU_THEN_CPU"
    )
    slope_raster.save(r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\gis_data\explore_data\explore_data.gdb\slope_raster")

In [11]:
#Convert slope raster to points
arcpy.env.overwriteOutput = True

arcpy.conversion.RasterToPoint(
    in_raster="slope_raster",
    out_point_features=r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\gis_data\explore_data\explore_data.gdb\slope_points",
    raster_field="Value"
)

#rename point field representing the slope to degrees
arcpy.management.AlterField(
    in_table="slope_points",
    field="grid_code",
    new_field_name="degrees",
    new_field_alias="degrees",
    field_type="",
    field_length=4,
    field_is_nullable="NULLABLE",
    clear_field_alias="DO_NOT_CLEAR"
)

<Result 'slope_points'>

In [12]:
#Convert elevation raster to points
arcpy.env.overwriteOutput = True

arcpy.conversion.RasterToPoint(
    in_raster="wetopo2022_albers_10k",
    out_point_features=r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\gis_data\explore_data\explore_data.gdb\etopo_points",
    raster_field="Value"
)

#rename field representing elevation to metres
arcpy.management.AlterField(
    in_table="etopo_points",
    field="grid_code",
    new_field_name="metres",
    new_field_alias="metres",
    field_type="",
    field_length=4,
    field_is_nullable="NULLABLE",
    clear_field_alias="DO_NOT_CLEAR"
)

<Result 'etopo_points'>

In [13]:
#Convert GAP status raster to points
arcpy.env.overwriteOutput = True

arcpy.conversion.RasterToPoint(
    in_raster="oil_states_protected_binary",
    out_point_features=r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\gis_data\explore_data\explore_data.gdb\protected_points",
    raster_field="Value"
)

#rename field representing binary protection status to protected
arcpy.management.AlterField(
    in_table="protected_points",
    field="grid_code",
    new_field_name="protected",
    new_field_alias="protected",
    field_type="",
    field_length=4,
    field_is_nullable="NULLABLE",
    clear_field_alias="DO_NOT_CLEAR"
)

<Result 'protected_points'>

In [14]:
#Join OGDP, Slope, Elevation, and Protection points where they overlap
arcpy.env.overwriteOutput = True

#OGDP with Slope
arcpy.analysis.SpatialJoin(
    target_features="ogdp50_raster_points",
    join_features="slope_points",
    out_feature_class=r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\gis_data\explore_data\explore_data.gdb\ogdp50_w_slope_points",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    field_mapping='grid_code "grid_code" true true false 4 Float 0 0,First,#,ogdp50_raster_points,grid_code,-1,-1;degrees "degrees" true true false 4 Float 0 0,First,#,slope_points,degrees,-1,-1',
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

#with Elevation
arcpy.analysis.SpatialJoin(
    target_features="ogdp50_w_slope_points",
    join_features="etopo_points",
    out_feature_class=r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\gis_data\explore_data\explore_data.gdb\ogdp_slope_etopo",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    field_mapping='grid_code "grid_code" true true false 4 Float 0 0,First,#,ogdp50_w_slope_points,grid_code,-1,-1;degrees "degrees" true true false 4 Float 0 0,First,#,ogdp50_w_slope_points,degrees,-1,-1;metres "metres" true true false 4 Float 0 0,First,#,etopo_points,metres,-1,-1',
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

#with Protections
arcpy.analysis.SpatialJoin(
    target_features="ogdp_slope_etopo",
    join_features="protected_points",
    out_feature_class=r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\gis_data\explore_data\explore_data.gdb\ogdp_slope_etopo_gap",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    field_mapping='grid_code "grid_code" true true false 4 Float 0 0,First,#,ogdp_slope_etopo,grid_code,-1,-1;degrees "degrees" true true false 4 Float 0 0,First,#,ogdp_slope_etopo,degrees,-1,-1;metres "metres" true true false 4 Float 0 0,First,#,ogdp_slope_etopo,metres,-1,-1;protected "protected" true true false 4 Long 0 0,First,#,protected_points,protected,-1,-1',
    match_option="INTERSECT",
    search_radius=None,
    distance_field_name="",
    match_fields=None
)

<Result 'C:\\Users\\MitchellChandler\\OneDrive - THE WILDERNESS SOCIETY\\OGDP model\\_Data\\gis_data\\explore_data\\explore_data.gdb\\ogdp_slope_etopo_gap'>

In [15]:
#Export as .csv
arcpy.env.overwriteOutput = True

arcpy.conversion.ExportTable(
    in_table="ogdp_slope_etopo_gap",
    out_table=r"C:\Users\MitchellChandler\OneDrive - THE WILDERNESS SOCIETY\OGDP model\_Data\scripts\results\ogdp_slope_etopo_gap.csv",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping='grid_code "grid_code" true true false 4 Float 0 0,First,#,ogdp_slope_etopo_gap,grid_code,-1,-1;degrees "degrees" true true false 4 Float 0 0,First,#,ogdp_slope_etopo_gap,degrees,-1,-1;metres "metres" true true false 4 Float 0 0,First,#,ogdp_slope_etopo_gap,metres,-1,-1;protected "protected" true true false 4 Long 0 0,First,#,ogdp_slope_etopo_gap,protected,-1,-1',
    sort_field=None
)

<Result 'C:\\Users\\MitchellChandler\\OneDrive - THE WILDERNESS SOCIETY\\OGDP model\\_Data\\scripts\\results\\ogdp_slope_etopo_gap.csv'>